# 🏛️ Enterprise AI Governance with IBM Granite
This notebook demonstrates how to run the open-source **IBM Granite 3.0** model locally on a free Google Colab T4 GPU, utilizing 4-bit quantization to prevent memory crashes.

We will simulate an Enterprise AI use case: summarizing a corporate compliance policy and identifying potential risks. In a real-world scenario, this process should be tied to an **AI Use Case** in *watsonx.governance* to maintain an auditable factsheet of model behavior.

## 1. Install Dependencies
We need `transformers` and `accelerate` to run the model, and `bitsandbytes` is the secret ingredient that allows us to run an 8-billion parameter model on a single free GPU by compressing it (quantization).

In [ ]:
!pip install -q transformers accelerate bitsandbytes

## 2. Configure Model and Quantization
Here we select the **Granite 3.0 8B Instruct** model. Because 8B parameters usually require more than 16GB of VRAM, we use `BitsAndBytesConfig` to load the model in 4-bit precision. This drastically reduces the memory footprint while maintaining performance.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Select the Granite model
model_id = "ibm-granite/granite-3.0-8b-instruct"

print("Downloading Granite (Quantized to save resources)...")

# Setup 4-bit quantization so Colab doesn't crash
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

## 3. Load Model and Tokenizer
We load the tokenizer and the model. We also explicitly set the `pad_token` to the `eos_token` (End of Sentence) to prevent formatting warnings during generation. The `device_map="auto"` argument ensures the model is intelligently placed on the GPU.

In [ ]:
# Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Ensure tokenizer has a pad token to avoid issues
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load Model
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

## 4. Define the Summarization Function
This function takes the raw corporate policy and wraps it in a Granite-compatible chat template. 
* We use a specific **System Prompt** to instruct the AI to act as a compliance assistant.
* We set a low `temperature` (0.2) to ensure the model produces factual, grounded responses rather than highly creative (hallucinated) text.

In [ ]:
def governed_summarize(text):
    # Standard Granite chat template structure
    messages = [
        {"role": "system", "content": "You are an enterprise AI assistant. Summarize the following corporate policy concisely, highlighting the main compliance risks."},
        {"role": "user", "content": text}
    ]

    # Apply chat template with return_dict=True to get input_ids and attention_mask
    inputs = tokenizer.apply_chat_template(
        messages, 
        add_generation_prompt=True, 
        return_tensors="pt",
        return_dict=True
    )
    
    # Move inputs to the model device (GPU)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate the summary
    outputs = model.generate(
        **inputs,
        max_new_tokens=250, 
        temperature=0.2,
        do_sample=True
    )
    
    # Extract only the newly generated text (ignoring the prompt)
    response = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(response, skip_special_tokens=True)

## 5. Execute the Enterprise Use Case
Let's test our governed model on a mock corporate Data Protection Policy.

In [ ]:
corporate_policy = """
Data Protection Policy v4.2:
All employees must ensure that sensitive customer data, including Personally Identifiable Information (PII), is encrypted at rest and in transit. Customer data must not be stored on local hard drives or unapproved third-party cloud services. In the event of a suspected data breach, employees must notify the IT Security Officer within 2 hours. Failure to comply may result in disciplinary action up to and including termination. AI tools used to process this data must be logged in the centralized watsonx governance registry.
"""

print("\n--- GRANITE SUMMARY ---")
print(governed_summarize(corporate_policy))